# Naive Bayes — Vietnamese Toxic Comment Detection

## Thông tin bài tập lớn

- **Môn học:** Trí tuệ nhân tạo — Học viện Công nghệ Bưu chính Viễn thông (PTIT)
- **Tài liệu:** Russell, S. & Norvig, P. (2021). *Artificial Intelligence: A Modern Approach* (4th Edition), Pearson Education.
- **Bài tập lớn:** Hệ thống phát hiện bình luận độc hại tiếng Việt.
- **Thành viên nhóm:**
  - Neural MLP: [đồng đội]
  - Naive Bayes: [em — phụ trách phần này]
- **Dataset:** ViHSD (Vietnamese Hate Speech Detection), 3 lớp: clean (0), hate (1), offensive (2).

## Mục tiêu notebook này

Trình bày lý thuyết và hiện thực Multinomial Naive Bayes (chương probabilistic reasoning — AIMA) để phân loại 3 lớp. Kết quả: lưu model artifact có thể dùng cho inference.

## 1. Lý thuyết: Multinomial Naive Bayes (AIMA, chương 13-20)

### 1.1. Bài toán phân loại Bayes tối ưu

Theo AIMA, **Bayes classifier tối ưu** chọn lớp có xác suất hậu nghiệm lớn nhất:

$$c^* = \arg\max_{c \in C} P(c \mid x_1, x_2, \dots, x_n)$$

Áp dụng **Bayes' theorem**:

$$P(c \mid \mathbf{x}) = \frac{P(c) \cdot P(\mathbf{x} \mid c)}{P(\mathbf{x})}$$

Vì $P(\mathbf{x})$ không phụ thuộc $c$, ta được:

$$c^* = \arg\max_{c} \underbrace{P(c)}_{\text{prior}} \cdot \underbrace{P(\mathbf{x} \mid c)}_{\text{likelihood}}$$

### 1.2. Naive Bayes assumption

$P(\mathbf{x} \mid c)$ rất khó ước lượng trực tiếp (curse of dimensionality). AIMA chấp nhận **naive conditional independence assumption**:

$$P(x_1, x_2, \dots, x_n \mid c) = \prod_{i=1}^{n} P(x_i \mid c)$$

Sai số do giả định này thường không tệ như tưởng tượng — AIMA Section 12.6 (trong 3rd) chứng minh nó vẫn là **optimal Bayes classifier** trong một số mô hình phái sinh (với class conditional independence đúng hoàn toàn).

### 1.3. Multinomial Naive Bayes cho text

Với document $\mathbf{x} = (x_1, x_2, \dots, x_n)$ gồm $n$ token, mỗi $x_i$ là word ID:

$$\hat{P}(c) = \frac{\text{count}(c)}{N}$$

$$\hat{P}(x_i \mid c) = \frac{\text{count}(x_i, c)}{\sum_{j \in V} \text{count}(x_j, c)}$$

**Laplace smoothing** (AIMA Section 12.6.1) tránh zero probability:

$$\hat{P}(x_i \mid c) = \frac{\text{count}(x_i, c) + \alpha}{\sum_{j \in V} \text{count}(x_j, c) + \alpha \cdot |V|}$$

Trong đó $\alpha \in [0, 1]$ là smoothing parameter, $|V|$ là vocabulary size.

### 1.4. Quyết định trong log-space

Để tránh underflow, ta dùng log-likelihood:

$$\hat{c} = \arg\max_c \left[ \log P(c) + \sum_{i=1}^{n} \log P(x_i \mid c) \right]$$

### 1.5. Áp dụng cho bài toán

- **3 lớp**: clean (0), hate (1), offensive (2).
- **Features**: bag-of-words (đếm token sau khi lowercase, tokenize, loại stopwords).
- **Imbalance**: prior bị skew (clean chiếm 83%) → cần SMOTE hoặc class_weight.

In [ ]:
# Cell 3 — Imports
import sys
from pathlib import Path
import json
import time
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    accuracy_score,
)
from imblearn.over_sampling import SMOTE

# Import preprocessing module của nhóm
sys.path.insert(0, str(Path('..').resolve() / 'src'))
from preprocessing import preprocess_text

# Paths
NB_DIR = Path('..').resolve()
ROOT_DIR = NB_DIR.parent
DATASET_DIR = ROOT_DIR / 'neuralMLP' / 'dataraw'
REPORTS_DIR = NB_DIR / 'reports'
MODEL_DIR = NB_DIR / 'model'

REPORTS_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)

print(f"[INFO] NB_DIR = {NB_DIR}")
print(f"[INFO] DATASET_DIR = {DATASET_DIR}")
print(f"[INFO] Has preprocess_text: {hasattr(preprocess_text, '__call__')}")

In [ ]:
# Cell 4 — Load ViHSD dataset
train_df = pd.read_csv(DATASET_DIR / 'ViHSD_train.csv')
val_df = pd.read_csv(DATASET_DIR / 'ViHSD_validation.csv')
test_df = pd.read_csv(DATASET_DIR / 'ViHSD_test.csv')

print(f"Train shape: {train_df.shape}")
print(f"Val shape:   {val_df.shape}")
print(f"Test shape:  {test_df.shape}")

print("\nLabel distribution:")
for name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    counts = df['label_id'].value_counts().sort_index()
    print(f"\n{name}:")
    for label, count in counts.items():
        pct = count / len(df) * 100
        print(f"  Label {label}: {count:>6} ({pct:5.2f}%)")

assert set(train_df['label_id'].unique()) == {0, 1, 2}, "Phải có đủ 3 lớp"
LABEL_MAP = {0: 'clean', 1: 'hate', 2: 'offensive'}
print(f"\nLabel map: {LABEL_MAP}")

In [ ]:
# Cell 5 — EDA (Exploratory Data Analysis) ngắn
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart phân phối
train_df['label_id'].value_counts().sort_index().plot(
    kind='bar', ax=axes[0], color=['#22c55e', '#ef4444', '#f59e0b']
)
axes[0].set_title('Phân phối lớp — Train set')
axes[0].set_xlabel('Label ID (0=clean, 1=hate, 2=offensive)')
axes[0].set_ylabel('Số lượng')

# Đếm tokens (sau khi lowercase + tách bằng space)
train_df['n_words'] = train_df['free_text'].str.split().str.len()
axes[1].hist(train_df['n_words'], bins=50, color='#3b82f6', edgecolor='black')
axes[1].set_title('Phân phối số từ/câu — Train set')
axes[1].set_xlabel('Số từ')
axes[1].set_ylabel('Tần suất')
axes[1].axvline(train_df['n_words'].median(), color='red', linestyle='--', label=f'Median: {train_df["n_words"].median():.0f}')
axes[1].legend()

plt.tight_layout()
plt.savefig(REPORTS_DIR / 'eda_distribution.png', dpi=120)
plt.show()

# In 2 câu mỗi class
print("\n" + "=" * 70)
print("MẪU CÂU THEO CLASS")
print("=" * 70)
for label in [0, 1, 2]:
    print(f"\n--- Label {label} ({LABEL_MAP[label]}) ---")
    samples = train_df[train_df['label_id'] == label]['free_text'].head(3).tolist()
    for s in samples:
        print(f"  • {s[:100]}{'...' if len(s) > 100 else ''}")